# Week 6 Exercise – The Price is Right (winniekariuki)

Moderate solution: predict product price from description using the course `pricer` package.

- **Data**: `ed-donner/items_lite` (public; no HF_TOKEN required)
- **Baselines**: constant (mean price) + LLM (GPT-4o-mini)
- **Evaluation**: `pricer.evaluator.evaluate()`
- **Gradio UI**: paste description → estimated price

Run from repo root or `week6/` so `pricer` imports work. Requires `OPENAI_API_KEY` in `.env`.

In [ ]:
import os
import re
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Add week6 to path for pricer import
for base in [Path.cwd(), Path.cwd() / "week6", Path.cwd().parent, Path.cwd().parent.parent]:
    if (base / "pricer" / "items.py").exists():
        sys.path.insert(0, str(base))
        break

from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI() if api_key else None
MODEL = "gpt-4o-mini"

In [ ]:
# Load data from HuggingFace
DATASET = "ed-donner/items_lite"
train, val, test = Item.from_hub(DATASET)

print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

# Mean training price for constant baseline
train_prices = [item.price for item in train]
MEAN_PRICE = sum(train_prices) / len(train_prices)
print(f"Mean training price: ${MEAN_PRICE:.2f}")

In [ ]:
# Baseline: always predict the mean price
def constant_pricer(item):
    return MEAN_PRICE

print("Evaluating constant baseline...")
evaluate(constant_pricer, test, size=100)

In [ ]:
# LLM pricer: GPT predicts price from item summary
SYSTEM_PROMPT = """You are a pricing expert. Given a product description, predict its price in USD.
Reply with ONLY a number. No dollar sign, no explanation. Just the number."""

def llm_pricer(item):
    if not client:
        return MEAN_PRICE
    text = item.summary or item.title or str(item)
    text = (text or "")[:800]
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Product: {text}\n\nPrice:"},
        ],
        temperature=0,
        max_tokens=20,
    )
    raw = (resp.choices[0].message.content or "").strip()
    match = re.search(r"[\d.]+", raw)
    return float(match.group()) if match else MEAN_PRICE

print("Evaluating LLM pricer...")
evaluate(llm_pricer, test, size=100)

In [ ]:
# Gradio UI
def estimate_price(description: str) -> str:
    if not description or not description.strip():
        return "Enter a product description."
    if not client:
        return "OpenAI API key not set."
    dummy = Item(title="", category="", price=0, summary=description.strip())
    pred = llm_pricer(dummy)
    return f"Estimated price: {pred:.2f}"

demo = gr.Interface(
    fn=estimate_price,
    inputs=gr.Textbox(
        label="Product description",
        lines=3,
        placeholder="e.g. Wireless Bluetooth earbuds, 24hr battery, noise cancelling",
    ),
    outputs=gr.Textbox(label="Estimated price"),
    title="Week 6 – Product Pricer",
    description="Predict price from product description using GPT-4o-mini.",
    examples=[
        ["Wireless Bluetooth earbuds, 24hr battery, noise cancelling"],
        ["Stainless steel water bottle 32oz insulated"],
        ["Mechanical keyboard RGB backlit, cherry MX switches"],
        ["Running shoes men's size 10 breathable mesh"],
        ["Noise cancelling over-ear headphones"],
        ["Coffee maker 12-cup programmable"],
        ["Smart watch fitness tracker with heart rate monitor"],
        ["Ceramic cookware set 10 piece"],
    ],
)

demo.launch()